# Reproducible ML Experiments

Fix seeds, establish a baseline, change one variable at a time, and save a results table.

In [ ]:
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
SEED=42
random.seed(SEED); np.random.seed(SEED)

## 1. Create a repeatable dataset and split

In [ ]:
X,y=make_classification(n_samples=1200,n_features=20,n_informative=8,n_redundant=4,weights=[.65,.35],random_state=SEED)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
print(X_train.shape,X_test.shape)

## 2. Define a common experiment function

In [ ]:
def run_experiment(name,C=1.0,class_weight=None,seed=SEED):
    model=make_pipeline(StandardScaler(),LogisticRegression(C=C,class_weight=class_weight,max_iter=2000,random_state=seed))
    model.fit(X_train,y_train); pred=model.predict(X_test)
    return {'run':name,'C':C,'class_weight':str(class_weight),'seed':seed,'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0)}

## 3. Baseline and regularization sweep

C is inverse regularization strength. Smaller C means stronger regularization. Change only C in this experiment.

In [ ]:
results=[run_experiment('baseline',C=1.0)]
for c in [.01,.1,1.,10.,100.]: results.append(run_experiment(f'C={c}',C=c))
results_df=pd.DataFrame(results).drop_duplicates(subset=['C','class_weight','seed']).reset_index(drop=True)
display(results_df.round(4))

## 4. Compare results

In [ ]:
plt.semilogx(results_df.C,results_df.f1,marker='o'); plt.xlabel('C (log scale)'); plt.ylabel('F1'); plt.title('Regularization sweep'); plt.grid(True); plt.show()

## 5. Save the experiment record

In [ ]:
results_df.to_csv('experiment_results.csv',index=False)
print('Saved experiment_results.csv')

## Exercises

1. Repeat with three split seeds and report mean/std F1.
2. Change only class_weight to 'balanced'.
3. Record package versions and run timestamp.
4. Explain why changing multiple variables at once weakens conclusions.
5. Why should test results not be used for repeated hyperparameter selection?